# 02 · Coleta e limpeza dos dados do INMET
**Projeto:** Dois biomas, dois regimes de fogo — Pantanal × Cerrado (2020–2023)

Este notebook transforma os dados horários de ~590 estações automáticas do INMET em uma tabela **diária**
(temperatura, umidade, chuva e dias sem chuva) e seleciona as estações que ficam **dentro ou perto**
do Pantanal e do Cerrado. O resultado alimenta a junção espacial com os focos de calor do INPE.

| Seção | O que faz |
|---|---|
| 1 | Configuração (Drive, parâmetros, caminhos) |
| 2 | Carga: copia os `.zip` anuais para o disco local |
| 3 | Funções de leitura, limpeza e agregação diária de **uma** estação |
| 4 | Processamento ano a ano (com checkpoint no Drive) |
| 5 | Consolidação e conferência de qualidade |
| 6 | Dias consecutivos sem chuva |
| 7 | Salva o diário nacional |
| 8 | Seleção das estações por bioma (limites do IBGE) e cobertura |
| 9 | Resumo, resultados de referência e notas para o README |

**Entradas:** `datasets_inmet/AAAA.zip` (um por ano, baixado de https://portal.inmet.gov.br/dadoshistoricos)
**Saídas** (em `datasets_inmet/processed/`): `inmet_diario_final.parquet`, `estacoes.csv`,
`estacoes_pantanal_cerrado.csv`, `inmet_diario_pantanal_cerrado.parquet`

**Se a sessão do Colab cair:** rode a Seção 1 e siga direto para a Seção 8, que recarrega o diário do Drive.
Na Seção 4, anos já processados são pulados (checkpoint).

**Memória:** cada estação é lida, limpa e **agregada por dia dentro do loop**. Em vez de ~20 milhões de linhas
horárias na RAM, guardamos ~600 mil linhas diárias.

## 1. Configuração

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import io
import os
import re
import glob
import shutil
import zipfile
import subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ---- parâmetros do projeto ----
ANOS = [2020, 2021, 2022, 2023]        # recorte temporal (2024 ficou fora)
BIOMAS_ALVO = ['cerrado', 'pantanal']  # biomas analisados
MARGEM_KM = 100                        # estações a até X km de um bioma entram na seleção

# ---- caminhos ----
PASTA_INMET = '/content/drive/MyDrive/datasets_inmet'   # onde ficam os .zip do INMET
PASTA_SAIDA = f'{PASTA_INMET}/processed'                # checkpoints e resultados
PASTA_BIOMAS = f'{PASTA_INMET}/biomas'                  # limites dos biomas (IBGE)
PASTA_LOCAL = '/content/inmet_zips'                     # cópia local rápida (disco do Colab)

for pasta in (PASTA_SAIDA, PASTA_BIOMAS, PASTA_LOCAL):
    os.makedirs(pasta, exist_ok=True)

## 2. Carga dos dados
Copiamos cada `.zip` do Drive para o disco local do Colab (1 arquivo grande é rápido; milhares de CSVs
pequenos lidos direto do Drive são muito lentos). Os CSVs são lidos **de dentro do zip**, sem extrair.

In [3]:
zips = {}
for origem in sorted(glob.glob(f'{PASTA_INMET}/*.zip')):
    achou = re.search(r'(\d{4})', os.path.basename(origem))
    if not achou or int(achou.group(1)) not in ANOS:
        continue
    ano = int(achou.group(1))
    destino = f'{PASTA_LOCAL}/{ano}.zip'
    if not os.path.exists(destino):
        shutil.copy(origem, destino)
    zips[ano] = destino

print("Anos encontrados:", sorted(zips))
assert sorted(zips) == ANOS, f"Faltam anos: {set(ANOS) - set(zips)}"

# quantas estações (CSVs) cada zip tem
n_zip = {}
for ano, caminho in zips.items():
    with zipfile.ZipFile(caminho) as zf:
        n_zip[ano] = sum(n.upper().endswith('.CSV') for n in zf.namelist())
print("Estações por zip:", n_zip)

Anos encontrados: [2020, 2021, 2022, 2023]
Estações por zip: {2020: 589, 2021: 588, 2022: 567, 2023: 567}


## 3. Funções de leitura, limpeza e agregação diária
`processar_estacao` recebe o conteúdo de **um** CSV (~8.700 linhas por ano) e devolve **uma linha por dia**:

1. lê os metadados do cabeçalho (código, UF, nome, latitude, longitude);
2. lê a tabela horária (`-9999` vira `NaN`; **não** preenchemos com 0, porque chuva 0 mm é diferente de "sem dado");
3. monta `DataHora` em **UTC** (mesmo padrão do BDQueimadas);
4. zera valores impossíveis (umidade fora de 0–100, temperatura fora de -10 a 50 °C, chuva negativa);
5. agrega por dia e mantém só dias com pelo menos **18 de 24 horas** de temperatura.

In [4]:
# nomes das colunas mudam um pouco de ano para ano: padronizamos pelo começo do texto
def renomear(col):
    c = col.upper()
    if c.startswith('DATA'): return 'Data'
    if c.startswith('HORA'): return 'Hora'
    if c.startswith('PRECIPITA'): return 'Precipitacao'
    if c.startswith('TEMPERATURA DO AR'): return 'Temperatura'
    if c.startswith('UMIDADE RELATIVA'): return 'Umidade'
    if c.startswith('VENTO, RAJADA'): return 'VentoRajada'
    return col


# lê só as colunas usadas (menos memória e mais rápido)
def coluna_util(nome):
    return nome.upper().startswith(('DATA', 'HORA', 'PRECIPITA', 'TEMPERATURA DO AR',
                                    'UMIDADE RELATIVA', 'VENTO, RAJADA'))


def processar_estacao(conteudo):
    # 1) metadados: 8 primeiras linhas (UF, estação, código, latitude, longitude...)
    linhas = conteudo.decode('latin-1').splitlines()[:8]
    meta = {}
    for linha in linhas:
        chave, _, valor = linha.partition(';')
        meta[chave.strip().rstrip(':').upper()] = valor.strip().strip(';')

    # 2) tabela horária (-9999 = sem medição -> NaN)
    df = pd.read_csv(io.BytesIO(conteudo), sep=';', decimal=',', skiprows=8,
                     encoding='latin-1', na_values=['-9999', '-9999.0'], usecols=coluna_util)
    df.columns = [renomear(c.strip()) for c in df.columns]
    for c in ['Precipitacao', 'Temperatura', 'Umidade', 'VentoRajada']:
        if c not in df.columns:
            df[c] = np.nan

    # 3) DataHora em UTC. 'Data': 'AAAA/MM/DD' ou 'AAAA-MM-DD' | 'Hora': '0000' ou '0000 UTC'
    #    (o pandas lê '0000' como o inteiro 0, por isso refazemos com zfill)
    data = pd.to_datetime(df['Data'].astype(str).str.strip().str.replace('/', '-'),
                          format='%Y-%m-%d', errors='coerce')
    hora = df['Hora'].astype(str).str.extract(r'(\d+)')[0].str.zfill(4).str[:2].astype(float)
    df['DataHora'] = data + pd.to_timedelta(hora, unit='h')
    df = df.dropna(subset=['DataHora']).drop_duplicates(subset='DataHora')

    # 4) valores fisicamente impossíveis viram NaN
    df.loc[~df['Umidade'].between(0, 100), 'Umidade'] = np.nan
    df.loc[~df['Temperatura'].between(-10, 50), 'Temperatura'] = np.nan
    df.loc[df['Precipitacao'] < 0, 'Precipitacao'] = np.nan

    # 5) agregação diária
    df['Dia'] = df['DataHora'].dt.floor('D')
    g = df.groupby('Dia')
    diario = pd.DataFrame({
        'Precip_total': g['Precipitacao'].sum(min_count=1),
        'Temp_max': g['Temperatura'].max(),
        'Temp_min': g['Temperatura'].min(),
        'Temp_med': g['Temperatura'].mean(),
        'Umid_min': g['Umidade'].min(),
        'Umid_med': g['Umidade'].mean(),
        'Vento_max': g['VentoRajada'].max(),
        'N_horas': g['Temperatura'].count(),
    }).reset_index().rename(columns={'Dia': 'Data'})

    # só dias com cobertura razoável (>= 18 de 24 horas)
    diario = diario[diario['N_horas'] >= 18].copy()

    # 6) metadados da estação
    diario['Codigo'] = str(meta.get('CODIGO (WMO)', '')).strip().upper()
    diario['UF'] = str(meta.get('UF', '')).strip().upper()
    diario['Estacao'] = str(meta.get('ESTACAO', '')).strip().upper()
    diario['Latitude'] = pd.to_numeric(meta.get('LATITUDE', '').replace(',', '.'), errors='coerce')
    diario['Longitude'] = pd.to_numeric(meta.get('LONGITUDE', '').replace(',', '.'), errors='coerce')

    # float32 para economizar memória
    for c in ['Precip_total', 'Temp_max', 'Temp_min', 'Temp_med', 'Umid_min', 'Umid_med', 'Vento_max']:
        diario[c] = diario[c].astype('float32')
    return diario

## 4. Processamento ano a ano
Cada ano vira um parquet no Drive (`inmet_diario_AAAA.parquet`). Se a sessão cair, anos prontos são pulados.
Para **reprocessar** um ano (por exemplo, depois de mudar uma função da Seção 3), apague o parquet dele.

In [5]:
for ano, caminho_zip in zips.items():
    arq_ano = f'{PASTA_SAIDA}/inmet_diario_{ano}.parquet'
    if os.path.exists(arq_ano):
        print(ano, 'já processado, pulando')
        continue

    partes, falhas = [], []
    with zipfile.ZipFile(caminho_zip) as zf:
        nomes = [n for n in zf.namelist() if n.upper().endswith('.CSV')]
        for nome in tqdm(nomes, desc=str(ano)):
            try:
                partes.append(processar_estacao(zf.read(nome)))
            except Exception as e:
                falhas.append((nome, str(e)))

    pd.concat(partes, ignore_index=True).to_parquet(arq_ano, index=False)
    print(f'{ano}: {len(partes)} estações ok, {len(falhas)} falhas')
    for nome, erro in falhas[:5]:
        print('   falhou:', nome, '->', erro)
    del partes

2020 já processado, pulando
2021 já processado, pulando
2022 já processado, pulando
2023 já processado, pulando


## 5. Consolidação e conferência de qualidade

In [6]:
arquivos_ano = sorted(glob.glob(f'{PASTA_SAIDA}/inmet_diario_[0-9][0-9][0-9][0-9].parquet'))
arquivos_ano = [f for f in arquivos_ano if int(re.search(r'(\d{4})', os.path.basename(f)).group(1)) in ANOS]
print([os.path.basename(f) for f in arquivos_ano])

diario = pd.concat([pd.read_parquet(f) for f in arquivos_ano], ignore_index=True)

# duplicados: mesma estação e mesmo dia
antes = len(diario)
diario = diario.drop_duplicates(subset=['Codigo', 'Data'])
print(f"Removidos {antes - len(diario)} duplicados de {antes} linhas")
print("Período:", diario['Data'].min().date(), '->', diario['Data'].max().date())
diario.info()

['inmet_diario_2020.parquet', 'inmet_diario_2021.parquet', 'inmet_diario_2022.parquet', 'inmet_diario_2023.parquet']
Removidos 0 duplicados de 583649 linhas
Período: 2020-01-01 -> 2023-12-31
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 583649 entries, 0 to 583648
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Data          583649 non-null  datetime64[ns]
 1   Precip_total  531545 non-null  float32       
 2   Temp_max      583649 non-null  float32       
 3   Temp_min      583649 non-null  float32       
 4   Temp_med      583649 non-null  float32       
 5   Umid_min      566506 non-null  float32       
 6   Umid_med      566506 non-null  float32       
 7   Vento_max     551471 non-null  float32       
 8   N_horas       583649 non-null  int64         
 9   Codigo        583649 non-null  object        
 10  UF            583649 non-null  object        
 11  Estacao       583649 non-null 

### 5.1 Estações por ano: arquivos do INMET × tabela diária
A diferença são estações que **não têm nenhum dia com 18+ horas de temperatura** naquele ano.
Em 2021 o número é bem maior; ao abrir exemplos, os arquivos dessas estações estavam praticamente vazios
no próprio INMET (sem leituras ou com poucas horas no ano). Ou seja, é um problema do dado, não do código.

In [7]:
n_diario = diario.groupby(diario['Data'].dt.year)['Codigo'].nunique()
tabela = pd.DataFrame({'estacoes_no_zip': pd.Series(n_zip), 'estacoes_no_diario': n_diario})
tabela['sem_dados_suficientes'] = tabela['estacoes_no_zip'] - tabela['estacoes_no_diario']
print(tabela.to_string())

      estacoes_no_zip  estacoes_no_diario  sem_dados_suficientes
2020              589                 558                     31
2021              588                 451                    137
2022              567                 526                     41
2023              567                 547                     20


### 5.2 Nulos, faixas de valores e cobertura de dias

In [8]:
print("Valores nulos por coluna (%):")
print((diario.isnull().mean() * 100).round(1).sort_values(ascending=False).head(6).to_string())

print("\nFaixas de valores:")
print(diario[['Precip_total', 'Temp_max', 'Temp_min', 'Umid_min', 'Vento_max']].describe().round(2).to_string())

print("\nDias com dados por estação-ano (máx. 365/366):")
ano_col = diario['Data'].dt.year.rename('Ano')
cobertura_dias = diario.groupby([ano_col, 'Codigo']).size().rename('dias')
print(cobertura_dias.describe().round(0).to_string())
print("Estação-ano com menos de 300 dias:", int((cobertura_dias < 300).sum()), "de", len(cobertura_dias))

Valores nulos por coluna (%):
Precip_total    8.9
Vento_max       5.5
Umid_med        2.9
Umid_min        2.9
Data            0.0
Temp_max        0.0

Faixas de valores:
       Precip_total   Temp_max   Temp_min   Umid_min  Vento_max
count     531545.00  583649.00  583649.00  566506.00  551471.00
mean           3.89      28.88      18.62      48.57       8.81
std           10.96       5.02       4.60      17.26       3.11
min            0.00      -1.40      -9.00       4.00       0.00
25%            0.00      26.10      16.10      35.00       6.90
50%            0.00      29.60      19.60      48.00       8.50
75%            1.40      32.30      22.10      61.00      10.40
max          284.60      44.70      35.00     100.00      50.00

Dias com dados por estação-ano (máx. 365/366):
count    2082.0
mean      280.0
std       108.0
min         1.0
25%       208.0
50%       339.0
75%       365.0
max       366.0
Estação-ano com menos de 300 dias: 827 de 2082


**O que já verificamos nos dados:**
- **Chuva máxima diária** (~285 mm) e **sequências sem chuva** (até ~214 dias, estação seca longa do Cerrado/Caatinga) são plausíveis.
- **Temperaturas negativas** vêm de estações de montanha (Itatiaia e Pico do Couto, RJ), onde gela de verdade — e ficam fora do recorte.
  O único valor claramente errado visto (Bragança/PA, -9 °C) também está fora do Pantanal e do Cerrado.
- **Muitas estações têm poucos dias** de dados (cerca de 40% das estações-ano têm menos de 300 dias). Na junção com os focos,
  a regra é usar a estação mais próxima **que tenha dado naquele dia**.

## 6. Dias consecutivos sem chuva
Calculado por estação, em ordem de data. Dia com **>= 1 mm** conta como chuvoso.

Regras para não inflar a contagem:
- Dia **sem medição de chuva** (`NaN`) não conta como seco: o valor do dia fica `NaN` e a contagem recomeça.
- **Lacunas de datas** (dias removidos por falta de dados) também reiniciam a contagem.
- Assim `Dias_sem_chuva` é uma contagem *conservadora* de dias secos **confirmados** seguidos.

Limitação: no começo de 2020 a contagem parte de zero, mesmo que a estação já estivesse seca antes.

In [9]:
diario = diario.sort_values(['Codigo', 'Data']).reset_index(drop=True)

# "corte" na contagem: choveu (>= 1 mm), chuva desconhecida (NaN) ou falta algum dia entre duas linhas
lacuna = diario.groupby('Codigo')['Data'].diff().dt.days.fillna(1) > 1
corte = (diario['Precip_total'] >= 1) | diario['Precip_total'].isna() | lacuna
bloco = corte.groupby(diario['Codigo']).cumsum()

seco = diario['Precip_total'] < 1
diario['Dias_sem_chuva'] = seco.groupby([diario['Codigo'], bloco]).cumsum().astype('float32')
diario.loc[diario['Precip_total'].isna(), 'Dias_sem_chuva'] = np.nan

print(diario.shape)
print(diario['Dias_sem_chuva'].describe().round(1).to_string())

(583649, 15)
count    531545.0
mean          9.6
std          19.5
min           0.0
25%           0.0
50%           3.0
75%           9.0
max         214.0


## 7. Salvando o diário nacional

In [10]:
estacoes = (diario[['Codigo', 'UF', 'Estacao', 'Latitude', 'Longitude']]
            .drop_duplicates('Codigo').reset_index(drop=True))

diario.to_parquet(f'{PASTA_SAIDA}/inmet_diario_final.parquet', index=False)
estacoes.to_csv(f'{PASTA_SAIDA}/estacoes.csv', index=False)
print(len(estacoes), 'estações |', diario.shape, 'linhas salvas em', PASTA_SAIDA)

590 estações | (583649, 15) linhas salvas em /content/drive/MyDrive/datasets_inmet/processed


## 8. Seleção das estações por bioma
Ficamos só com estações **dentro ou a até `MARGEM_KM` km** do Pantanal e do Cerrado.

- **Limites dos biomas:** IBGE, versão 2025, escala 1:250.000 (arquivo `dmRgNtTrBmBr_v2025_vetor_e250K.zip`).
  Usamos a camada de **domínios**, cuja coluna `nm_bm` traz o bioma; a união dos polígonos de cada bioma forma o contorno dele.
- Distâncias calculadas na projeção **EPSG:5880** (SIRGAS 2000 / Brazil Polyconic, em metros).

*Esta seção pode ser executada sozinha depois da Seção 1, caso a sessão tenha caído.*

### 8.1 Recarregar o diário (se necessário) e baixar os limites dos biomas

In [11]:
if 'diario' not in globals():
    diario = pd.read_parquet(f'{PASTA_SAIDA}/inmet_diario_final.parquet')
    print('diário recarregado do Drive:', diario.shape)

URL_BIOMAS = ('https://geoftp.ibge.gov.br/informacoes_ambientais/estudos_ambientais/biomas/'
              'dominios_regioes_naturais_do_brasil/2025/dados_geoespaciais/'
              'dmRgNtTrBmBr_v2025_vetor_e250K.zip')
ZIP_BIOMAS = f'{PASTA_BIOMAS}/dmRgNtTrBmBr_v2025_vetor_e250K.zip'

if not os.path.exists(ZIP_BIOMAS):
    subprocess.run(['wget', '-q', '-O', ZIP_BIOMAS, URL_BIOMAS], check=True)
print(round(os.path.getsize(ZIP_BIOMAS) / 1e6, 1), 'MB em', ZIP_BIOMAS)

52.6 MB em /content/drive/MyDrive/datasets_inmet/biomas/dmRgNtTrBmBr_v2025_vetor_e250K.zip


### 8.2 Ler os polígonos dos biomas

In [12]:
import geopandas as gpd

os.makedirs('/content/biomas', exist_ok=True)
with zipfile.ZipFile(ZIP_BIOMAS) as zf:
    zf.extractall('/content/biomas')

shps = sorted(glob.glob('/content/biomas/**/*.shp', recursive=True))
print([os.path.basename(s) for s in shps])
shp = next((s for s in shps if 'dominios' in os.path.basename(s).lower()), shps[0])
print('usando:', os.path.basename(shp))

biomas = gpd.read_file(shp)
if biomas.crs is None:
    biomas = biomas.set_crs('EPSG:4674')          # SIRGAS 2000, padrão do IBGE
biomas = biomas.to_crs('EPSG:5880')               # projeção em metros para o Brasil

# acha sozinho a coluna que traz o nome do bioma
colunas = [c for c in biomas.columns if c != biomas.geometry.name]
COL_BIOMA = next(c for c in colunas
                 if biomas[c].astype(str).str.strip().str.lower().eq('cerrado').any())
biomas['_nome'] = biomas[COL_BIOMA].astype(str).str.strip().str.lower()
print('coluna do bioma:', COL_BIOMA, '->', sorted(biomas[COL_BIOMA].unique()))


def poligono_do_bioma(nome):
    # une os polígonos (um por domínio) de um bioma
    parte = biomas.loc[biomas['_nome'] == nome, 'geometry']
    return parte.union_all() if hasattr(parte, 'union_all') else parte.unary_union

['dmRgNtTrBmBr_v2025_dominios_vetor_e250K.shp', 'dmRgNtTrBmBr_v2025_regioes_vetor_e250K.shp']
usando: dmRgNtTrBmBr_v2025_dominios_vetor_e250K.shp
coluna do bioma: nm_bm -> ['Amazônia', 'Caatinga', 'Cerrado', 'Ilhas Oceânicas', 'Mata Atlântica', 'Pampa', 'Pantanal']


### 8.3 Distância de cada estação até cada bioma e seleção

In [13]:
estacoes = (diario[['Codigo', 'UF', 'Estacao', 'Latitude', 'Longitude']]
            .drop_duplicates('Codigo').reset_index(drop=True))

est = gpd.GeoDataFrame(estacoes,
        geometry=gpd.points_from_xy(estacoes['Longitude'], estacoes['Latitude']),
        crs='EPSG:4326').to_crs('EPSG:5880')

# distância (km) até cada bioma (0 = dentro)
for nome in BIOMAS_ALVO:
    est[f'dist_{nome}_km'] = est.distance(poligono_do_bioma(nome)) / 1000

colunas_dist = [f'dist_{nome}_km' for nome in BIOMAS_ALVO]
sel = est[(est[colunas_dist] <= MARGEM_KM).any(axis=1)]

print(len(sel), 'estações selecionadas de', len(est), f'(margem de {MARGEM_KM} km)')
for nome in BIOMAS_ALVO:
    print(f'  dentro do {nome}:', int((est[f'dist_{nome}_km'] == 0).sum()),
          '| a até 50/100/150 km:', [int((est[f'dist_{nome}_km'] <= k).sum()) for k in (50, 100, 150)])

print('\nEstações dentro do Pantanal:')
print(est.loc[est['dist_pantanal_km'] == 0, ['Codigo', 'UF', 'Estacao']].to_string(index=False))

diario_sel = diario[diario['Codigo'].isin(sel['Codigo'])]
print('\nDiário das estações selecionadas:', diario_sel.shape)

277 estações selecionadas de 590 (margem de 100 km)
  dentro do cerrado: 169 | a até 50/100/150 km: [240, 276, 306]
  dentro do pantanal: 4 | a até 50/100/150 km: [12, 19, 29]

Estações dentro do Pantanal:
Codigo UF        Estacao
  A717 MS       NHUMIRIM
  A723 MS PORTO MURTINHO
  A724 MS        CORUMBA
  A941 MT        CACERES

Diário das estações selecionadas: (277891, 15)


### 8.4 Cobertura: a que distância fica a estação mais próxima?
Espalhamos uma grade de pontos dentro de cada bioma e medimos a distância até a estação mais próxima.
Serve para **escolher o raio** da junção: se muita área do bioma fica longe de qualquer estação, o clima associado
aos focos dali é menos confiável. (Mede cobertura por **área**; os focos não se distribuem uniformemente.)

In [14]:
def cobertura(nome, passo_km, limites_km=(25, 50, 100, 150)):
    poligono = poligono_do_bioma(nome)
    x0, y0, x1, y1 = poligono.bounds
    xx, yy = np.meshgrid(np.arange(x0, x1, passo_km * 1000), np.arange(y0, y1, passo_km * 1000))
    pontos = gpd.GeoSeries(gpd.points_from_xy(xx.ravel(), yy.ravel()), crs=biomas.crs)
    dentro = pontos[pontos.within(poligono)]
    p = np.c_[dentro.x, dentro.y]
    e = np.c_[est.geometry.x, est.geometry.y]
    d = np.sqrt(((p[:, None, :] - e[None, :, :]) ** 2).sum(axis=2)).min(axis=1) / 1000
    print(f'\n{nome.upper()}: {len(p)} pontos de grade ({passo_km} km)')
    for lim in limites_km:
        print(f'  até {lim:>3} km de uma estação: {(d <= lim).mean() * 100:5.1f}% da área')
    print(f'  distância mediana até a estação mais próxima: {np.median(d):.0f} km')


cobertura('pantanal', passo_km=10)
cobertura('cerrado', passo_km=25)


PANTANAL: 1506 pontos de grade (10 km)
  até  25 km de uma estação:   4.9% da área
  até  50 km de uma estação:  18.3% da área
  até 100 km de uma estação:  63.0% da área
  até 150 km de uma estação:  93.4% da área
  distância mediana até a estação mais próxima: 86 km

CERRADO: 3219 pontos de grade (25 km)
  até  25 km de uma estação:  16.1% da área
  até  50 km de uma estação:  53.8% da área
  até 100 km de uma estação:  97.1% da área
  até 150 km de uma estação:  99.9% da área
  distância mediana até a estação mais próxima: 47 km


### 8.5 Salvar as estações e o diário selecionados (entrada da junção espacial)

In [15]:
cols_saida = ['Codigo', 'UF', 'Estacao', 'Latitude', 'Longitude'] + colunas_dist
sel[cols_saida].to_csv(f'{PASTA_SAIDA}/estacoes_pantanal_cerrado.csv', index=False)
diario_sel.to_parquet(f'{PASTA_SAIDA}/inmet_diario_pantanal_cerrado.parquet', index=False)
print('Salvo:', len(sel), 'estações |', diario_sel.shape, 'linhas em', PASTA_SAIDA)

Salvo: 277 estações | (277891, 15) linhas em /content/drive/MyDrive/datasets_inmet/processed


## 9. Resumo e próximos passos

**Resultados de referência** (execução anterior; conferem se uma nova execução está igual):

| Item | Valor |
|---|---|
| Estações nos zips (2020 / 2021 / 2022 / 2023) | 589 / 588 / 567 / 567 |
| Estações no diário (2020 / 2021 / 2022 / 2023) | 558 / 451 / 526 / 547 |
| Linhas do diário nacional | 583.649 (590 estações) |
| Estações selecionadas (margem de 100 km) | **277** de 590 |
| Dentro do Cerrado / dentro do Pantanal | **169 / 4** |
| Estações dentro do Pantanal | A717 Nhumirim, A723 Porto Murtinho, A724 Corumbá, A941 Cáceres |
| Estações a até 50 / 100 / 150 km do Pantanal | 12 / 19 / 29 |
| Linhas do diário selecionado | 277.891 |
| Área do Pantanal a até 50 / 100 / 150 km de uma estação | 18,3% / 63,0% / 93,4% |
| Área do Cerrado a até 50 / 100 / 150 km de uma estação | 53,8% / 97,1% / 99,9% |

**Para o README (seções de dados, metodologia e limitações):**
- Fonte do INMET: dados históricos, arquivos anuais 2020–2023, estações automáticas (horário em UTC).
- Limites dos biomas: IBGE, versão 2025, escala 1:250.000, arquivo `dmRgNtTrBmBr_v2025_dominios_vetor_e250K.shp`, coluna `nm_bm`.
- Raio: 100 km como principal; 50 km como teste de sensibilidade. A cobertura do Pantanal é bem pior que a do Cerrado
  (63% da área a até 100 km de uma estação, contra 97%), e só 4 estações ficam dentro dele.
- Muitas estações têm poucos dias de dados, e `Dias_sem_chuva` é uma contagem conservadora (reinicia após dia sem medição de chuva).
- O bioma de cada análise é o do **foco** (campo `Bioma` do INPE), não o da estação.

**Próximo notebook:** `03_juncao_espacial` — para cada foco, a estação mais próxima **com dado no dia**, dentro do raio.